# **Homework 4 - Adversarial Attack**

## Enviroment & Download

We make use of [pytorchcv](https://pypi.org/project/pytorchcv/) to obtain CIFAR-10 pretrained model, so we need to set up the enviroment first. We also need to download the data (200 images) which we want to attack.

In [ ]:
# set up environment
!pip install pytorchcv
# download
!gdown --id 1fHi1ko7wr80wXkXpqpqpOxuYH1mClXoX -O data.zip
# unzip
!unzip ./data.zip
!rm ./data.zip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.2/134.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 585.2/585.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 48.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink

## Global Settings (10 pts)

* $\epsilon$ is fixed to be 8. But on **Data section**, we will first apply transforms on raw pixel value (0-255 scale) **by ToTensor (to 0-1 scale)** and then **Normalize (subtract mean divide std)**. $\epsilon$ should be set to $\frac{8}{255 * std}$ during attack.

* Explaination (optional)
    * Denote the first pixel of original image as $p$, and the first pixel of adversarial image as $a$.
    * The $\epsilon$ constraints tell us $\left| p-a \right| <= 8$.
    * ToTensor() can be seen as a function where $T(x) = x/255$.
    * Normalize() can be seen as a function where $N(x) = (x-mean)/std$ where $mean$ and $std$ are constants.
    * After applying ToTensor() and Normalize() on $p$ and $a$, the constraint becomes $\left| N(T(p))-N(T(a)) \right| = \left| \frac{\frac{p}{255}-mean}{std}-\frac{\frac{a}{255}-mean}{std} \right| = \frac{1}{255 * std} \left| p-a \right| <= \frac{8}{255 * std}.$
    * So, we should set $\epsilon$ to be $\frac{8}{255 * std}$ after ToTensor() and Normalize().

In [ ]:
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 8

# TODO: the mean and std are the calculated statistics from cifar_10 dataset
cifar_10_mean = (?????) # mean for the three channels of cifar_10 images
cifar_10_std = (?????) # std for the three channels of cifar_10 images

# convert mean and std to 3-dimensional tensors for future operations
mean = torch.tensor(cifar_10_mean).to(device).view(3, 1, 1)
std = torch.tensor(cifar_10_std).to(device).view(3, 1, 1)

# TODO: Set the epsilon value as introduced above
epsilon = ?????

# alpha (step size) can be decided by yourself
alpha = 0.8/255/std

root = './data' # directory for storing benign images
# benign images: images which do not contain adversarial perturbations
# adversarial images: images which include adversarial perturbations

# valid normalized image range corresponding to raw pixel range [0, 1]
lower_limit = ((0 - mean) / std)
upper_limit = ((1 - mean) / std)


## Data (10 pts)

Construct dataset and dataloader from root directory. Note that we store the filename of each image for future usage.

In [ ]:
import os
import glob
import shutil
import numpy as np
from PIL import Image
from torchvision.transforms import transforms
from torch.utils.data import Dataset, DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    # TODO: Use normilize from transforms to normalize data(with cifar10 mean and std)
    ?????
])

class AdvDataset(Dataset):
    def __init__(self, data_dir, transform):
        self.images = []
        self.labels = []
        self.names = []
        '''
        data_dir
        ├── class_dir
        │   ├── class1.png
        │   ├── ...
        │   ├── class20.png
        '''
        for i, class_dir in enumerate(sorted(glob.glob(f'{data_dir}/*'))):
            images = sorted(glob.glob(f'{class_dir}/*'))
            self.images += images
            self.labels += ([i] * len(images))
            self.names += [os.path.relpath(imgs, data_dir) for imgs in images]
        self.transform = transform
    def __getitem__(self, idx):
        image = self.transform(Image.open(self.images[idx]))
        label = self.labels[idx]
        return image, label
    def __getname__(self):
        return self.names
    def __len__(self):
        return len(self.images)

adv_set = AdvDataset(root, transform=transform)
adv_names = adv_set.__getname__()

## Apply dataloader to adv_set with batchsize=batchsize, shuffle=False
adv_loader = ?????

print(f'number of images = {adv_set.__len__()}')

number of images = 200


## Utils -- Benign Images Evaluation (5 pt)

In [ ]:
# To evaluate the performance of model on benign images
def epoch_benign(model, loader, loss_fn):
    # TODO: set the model to the correct mode for evaluation (hint: affects BatchNorm/Dropout)
    model.?????

    train_acc, train_loss = 0.0, 0.0

    for x, y in loader:
        # Move data to the correct device
        x, y = x.to(device), y.to(device)

        # TODO: Inference only — avoid unnecessary gradient tracking
        with torch.?????():
            # Forward pass
            yp = model(x)
            loss = loss_fn(yp, y)

        # Calculate number of correct predictions in this batch
        correct_preds = (yp.argmax(dim=1) == y).sum().item()

        # Update running totals
        train_acc += correct_preds
        train_loss += loss.item() * x.shape[0]  # Scale loss back to batch level

    # TODO: return average accuracy and average loss over the dataset
    # Question: what should you divide by?
    return ?????


## Utils -- Attack Algorithm (20 pts)

In [ ]:
# perform fgsm attack
def fgsm(model, x, y, loss_fn, epsilon=epsilon):
    x_adv = x.detach().clone()  # initialize x_adv as original benign image x
    x_adv.requires_grad_(True)

    model.zero_grad()
    yp = model(x_adv)
    loss = loss_fn(yp, y)
    loss.backward()

    # fgsm: use gradient ascent on x_adv to maximize loss
    x_adv = x_adv + epsilon * x_adv.grad.detach().sign()
    x_adv = torch.max(torch.min(x_adv, upper_limit), lower_limit)
    return x_adv.detach()


# perform iterative fgsm attack
def ifgsm(model, x, y, loss_fn, epsilon=epsilon, alpha=alpha, num_iter=10):
    x_adv = x.detach().clone()
    x_orig = x.detach().clone()

    for _ in range(num_iter):
        x_adv.requires_grad_(True)

        model.zero_grad()
        yp = model(x_adv)
        loss = loss_fn(yp, y)
        loss.backward()

        # iterative gradient ascent with sign update
        x_adv = x_adv + alpha * x_adv.grad.detach().sign()

        # project adversarial examples back to the epsilon-ball around the original image
        x_adv = torch.max(torch.min(x_adv, x_orig + epsilon), x_orig - epsilon)

        # clip to valid normalized image range
        x_adv = torch.max(torch.min(x_adv, upper_limit), lower_limit).detach()

    return x_adv


# helper to attack a target model with one or more proxy models
def transfer_attack(attack_models, x, y, loss_fn, attack_fn=fgsm):
    if not isinstance(attack_models, (list, tuple)):
        attack_models = [attack_models]

    # use the average gradient/logit signal from the proxy models
    def ensemble_attack(dummy_model, xb, yb, lf):
        x_adv = xb.detach().clone()
        x_adv.requires_grad_(True)

        total_loss = 0.0
        for proxy in attack_models:
            proxy.zero_grad()
            logits = proxy(x_adv)
            total_loss = total_loss + lf(logits, yb)

        total_loss = total_loss / len(attack_models)
        total_loss.backward()

        x_adv = x_adv + epsilon * x_adv.grad.detach().sign()
        x_adv = torch.max(torch.min(x_adv, upper_limit), lower_limit)
        return x_adv.detach()

    return ensemble_attack(None, x, y, loss_fn)

## Utils -- Attack (15 pts)

* Recall
    * ToTensor() can be seen as a function where $T(x) = x/255$.
    * Normalize() can be seen as a function where $N(x) = (x-mean)/std$ where $mean$ and $std$ are constants.

* Inverse function
    * Inverse Normalize() can be seen as a function where $N^{-1}(x) = x*std+mean$ where $mean$ and $std$ are constants.
    * Inverse ToTensor() can be seen as a function where $T^{-1}(x) = x*255$.

* Special Noted
    * ToTensor() will also convert the image from shape (height, width, channel) to shape (channel, height, width), so we also need to transpose the shape back to original shape.
    * Since our dataloader samples a batch of data, what we need here is to transpose **(batch_size, channel, height, width)** back to **(batch_size, height, width, channel)** using np.transpose.

In [ ]:
# perform adversarial attack and generate adversarial examples
def gen_adv_examples(model, loader, attack, loss_fn):
    model.eval()
    adv_names = []
    train_acc, train_loss = 0.0, 0.0
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        x_adv = attack(model, x, y, loss_fn) # obtain adversarial examples
        yp = model(x_adv)
        loss = loss_fn(yp, y)
        # TODO: Compute batch accuracy and add to running total
        train_acc ?????
        train_loss ?????

        # store adversarial examples
        adv_ex = ((x_adv) * std + mean).clamp(0, 1) # to 0-1 scale
        adv_ex = (adv_ex * 255).clamp(0, 255) # 0-255 scale
        adv_ex = adv_ex.detach().cpu().data.numpy().round() # round to remove decimal part

        ## TODO: Apply transpose to adv_ex to transpose (bs, C, H, W) back to (bs, H, W, C)
        adv_ex = ?????
        # TODO: Accumulate all adv_ex into a single array across batches
        adv_examples = ?????
    return adv_examples, train_acc / len(loader.dataset), train_loss / len(loader.dataset)

# create directory which stores adversarial examples
def create_dir(data_dir, adv_dir, adv_examples, adv_names):
    if os.path.exists(adv_dir) is not True:
        _ = shutil.copytree(data_dir, adv_dir)
    # TODO: Save each image (adv_examples) to corresponding path in adv_dir using the filenames from adv_names
    # Note: image pixel value should be unsigned int

## Utils -- Transfer / Black-Box Evaluation

For black-box evaluation, adversarial examples are generated on one or more **proxy models** and then evaluated on the fixed **target model**. This matches the assignment requirement that the attack should be performed in a black-box setting using proxy networks. fileciteturn6file1turn6file2

In [ ]:
# generate adversarial examples on proxy model(s), then evaluate on a different target model
def gen_transfer_adv_examples(target_model, loader, attack_fn, loss_fn):
    target_model.eval()
    adv_examples = []

    train_acc, train_loss = 0.0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        x_adv = attack_fn(x, y)
        yp = target_model(x_adv)
        loss = loss_fn(yp, y)

        train_acc += (yp.argmax(dim=1) == y).sum().item()
        train_loss += loss.item() * x.shape[0]

        adv_ex = ((x_adv) * std + mean).clamp(0, 1)
        adv_ex = (adv_ex * 255).clamp(0, 255)
        adv_ex = adv_ex.detach().cpu().numpy().round()
        adv_ex = np.transpose(adv_ex, (0, 2, 3, 1)).astype(np.uint8)
        adv_examples.append(adv_ex)

    adv_examples = np.concatenate(adv_examples, axis=0)
    return adv_examples, train_acc / len(loader.dataset), train_loss / len(loader.dataset)

## Model / Loss Function (5 pt)

In [ ]:
from pytorchcv.model_provider import get_model as ptcv_get_model

model = ptcv_get_model('resnet110_cifar10', pretrained=True).to(device)
## TODO: Apply crossentropyloss
loss_fn = ?????

benign_acc, benign_loss = epoch_benign(model, adv_loader, loss_fn)
print(f'benign_acc = {benign_acc:.5f}, benign_loss = {benign_loss:.5f}')

benign_acc = 0.95000, benign_loss = 0.22678


## FGSM (10 pts)

In [ ]:
# TODO: Apply gen_adv_examples to (model, adv_loader, fgsm, loss_fn)
adv_examples, fgsm_acc, fgsm_loss = ?????
print(f'fgsm_acc = {fgsm_acc:.5f}, fgsm_loss = {fgsm_loss:.5f}')

create_dir(root, 'fgsm', adv_examples, adv_names)

fgsm_acc = 0.59000, fgsm_loss = 2.49272


## I-FGSM

In [ ]:
num_iter = 10

ifgsm_examples, ifgsm_acc, ifgsm_loss = gen_adv_examples(
    model,
    adv_loader,
    lambda m, x, y, loss_fn: ifgsm(m, x, y, loss_fn, epsilon=epsilon, alpha=alpha, num_iter=num_iter),
    loss_fn
)
print(f'ifgsm_acc = {ifgsm_acc:.5f}, ifgsm_loss = {ifgsm_loss:.5f}')

ifgsm_dir = os.path.join(save_root, 'ifgsm')
create_dir(root, ifgsm_dir, ifgsm_examples, adv_names)
np.save(os.path.join(save_root, 'ifgsm_examples.npy'), ifgsm_examples)
print(f'Saved I-FGSM adversarial images to: {ifgsm_dir}')

## Black-Box / Transfer Attack

In [ ]:
# target model: fixed victim model
target_model = model

# proxy models used to craft black-box transferable adversarial examples
proxy_model_1 = ptcv_get_model('resnet56_cifar10', pretrained=True).to(device)
proxy_model_2 = ptcv_get_model('preresnet56_cifar10', pretrained=True).to(device)

for proxy in [proxy_model_1, proxy_model_2]:
    proxy.eval()

# single-proxy FGSM transfer attack
single_proxy_attack = lambda x, y: fgsm(proxy_model_1, x, y, loss_fn)
bb_fgsm_examples, bb_fgsm_acc, bb_fgsm_loss = gen_transfer_adv_examples(
    target_model, adv_loader, single_proxy_attack, loss_fn
)
print(f'black_box_fgsm_acc = {bb_fgsm_acc:.5f}, black_box_fgsm_loss = {bb_fgsm_loss:.5f}')

bb_fgsm_dir = os.path.join(save_root, 'black_box_fgsm')
create_dir(root, bb_fgsm_dir, bb_fgsm_examples, adv_names)
np.save(os.path.join(save_root, 'black_box_fgsm_examples.npy'), bb_fgsm_examples)
print(f'Saved black-box FGSM adversarial images to: {bb_fgsm_dir}')

# ensemble-proxy FGSM transfer attack (stronger baseline)
ensemble_proxy_attack = lambda x, y: transfer_attack([proxy_model_1, proxy_model_2], x, y, loss_fn, attack_fn=fgsm)
bb_ensemble_examples, bb_ensemble_acc, bb_ensemble_loss = gen_transfer_adv_examples(
    target_model, adv_loader, ensemble_proxy_attack, loss_fn
)
print(f'black_box_ensemble_acc = {bb_ensemble_acc:.5f}, black_box_ensemble_loss = {bb_ensemble_loss:.5f}')

bb_ensemble_dir = os.path.join(save_root, 'black_box_ensemble')
create_dir(root, bb_ensemble_dir, bb_ensemble_examples, adv_names)
np.save(os.path.join(save_root, 'black_box_ensemble_examples.npy'), bb_ensemble_examples)
print(f'Saved black-box ensemble adversarial images to: {bb_ensemble_dir}')

## Compress the images

In [ ]:
import tarfile

attack_dirs = {
    'fgsm': os.path.join(save_root, 'fgsm'),
    'ifgsm': os.path.join(save_root, 'ifgsm'),
    'black_box_fgsm': os.path.join(save_root, 'black_box_fgsm'),
    'black_box_ensemble': os.path.join(save_root, 'black_box_ensemble'),
}

for attack_name, attack_dir in attack_dirs.items():
    archive_path = os.path.join(save_root, f'{attack_name}.tgz')
    with tarfile.open(archive_path, 'w:gz') as tar:
        tar.add(attack_dir, arcname=attack_name)
    print(f'Compressed {attack_name} images to: {archive_path}')

## Visualization (5 Points)

In [ ]:
import matplotlib.pyplot as plt

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

def visualize_attack_result(folder_name, output_name):
    plt.figure(figsize=(10, 20))
    cnt = 0
    for cls_name in classes:
        path = f'{cls_name}/{cls_name}1.png'

        # benign image
        cnt += 1
        plt.subplot(len(classes), 2, cnt)
        im = Image.open(f'./data/{path}')
        logit = model(transform(im).unsqueeze(0).to(device))[0]
        predict = logit.argmax(-1).item()
        prob = logit.softmax(-1)[predict].item()
        plt.title(f'benign: {cls_name}1.png\n{classes[predict]}: {prob:.2%}')
        plt.axis('off')
        plt.imshow(np.array(im))

        # adversarial image
        cnt += 1
        plt.subplot(len(classes), 2, cnt)
        im = Image.open(os.path.join(save_root, folder_name, path))
        logit = model(transform(im).unsqueeze(0).to(device))[0]
        predict = logit.argmax(-1).item()
        prob = logit.softmax(-1)[predict].item()
        plt.title(f'{folder_name}: {cls_name}1.png\n{classes[predict]}: {prob:.2%}')
        plt.axis('off')
        plt.imshow(np.array(im))

    plt.tight_layout()
    viz_path = os.path.join(save_root, output_name)
    plt.savefig(viz_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved visualization to: {viz_path}')

visualize_attack_result('fgsm', 'fgsm_visualization.png')
visualize_attack_result('ifgsm', 'ifgsm_visualization.png')
visualize_attack_result('black_box_fgsm', 'black_box_fgsm_visualization.png')
visualize_attack_result('black_box_ensemble', 'black_box_ensemble_visualization.png')

## Reference

Course materials and homework instructions were used to determine the required attack setting, including non-targeted FGSM, I-FGSM, black-box attack through proxy networks, fixed \(\epsilon=8\), and \(L_\infty\) evaluation. fileciteturn6file1turn6file2